In [1]:
!git clone https://github.com/vfxbymani/ai-localization-pipeline.git
%cd ai-localization-pipeline
!ls

Cloning into 'ai-localization-pipeline'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 43 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 1.30 MiB | 6.77 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/kaggle/working/ai-localization-pipeline
day01-pytorch-basics.ipynb	   day05-whisper-wer.ipynb   test_video.srt
day02-transfer-learning.ipynb	   day06-video-to-srt.ipynb  video_to_srt.py
day03-huggingface-basics.ipynb	   LOG.md
day04-whisper-transcription.ipynb  README.md


In [2]:
import glob
import os
import subprocess

mp3 = glob.glob("/kaggle/input/**/*.mp3", recursive=True)[0]
print("audio file:", mp3)

subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-f", "lavfi", "-i", "color=c=black:s=640x360:r=10",
    "-i", mp3,
    "-shortest", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac",
    "tamil_test_video.mp4",
], check=True)

print("tamil_test_video.mp4 size (KB):", os.path.getsize("tamil_test_video.mp4") // 1024)

audio file: /kaggle/input/datasets/vfxbymani/tamil-clip/Pachaimamalai (mp3cut.net).mp3
tamil_test_video.mp4 size (KB): 373


In [3]:
!python video_to_srt.py tamil_test_video.mp4 -o tamil_small.srt --language ta --model openai/whisper-small
!python video_to_srt.py tamil_test_video.mp4 -o tamil_medium.srt --language ta --model openai/whisper-medium

print("----- small -----")
!cat tamil_small.srt
print("----- medium -----")
!cat tamil_medium.srt

preprocessor_config.json: 100%|███████████████| 185k/185k [00:00<00:00, 249MB/s]
config.json: 100%|█████████████████████████| 1.97k/1.97k [00:00<00:00, 4.79MB/s]
tokenizer_config.json: 100%|██████████████████| 283k/283k [00:00<00:00, 286MB/s]
vocab.json: 100%|████████████████████████████| 836k/836k [00:00<00:00, 76.6MB/s]
tokenizer.json: 100%|███████████████████████| 2.48M/2.48M [00:00<00:00, 127MB/s]
merges.txt: 100%|████████████████████████████| 494k/494k [00:00<00:00, 75.5MB/s]
normalizer.json: 100%|█████████████████████| 52.7k/52.7k [00:00<00:00, 96.6MB/s]
added_tokens.json: 100%|███████████████████| 34.6k/34.6k [00:00<00:00, 54.2MB/s]
special_tokens_map.json: 100%|█████████████| 2.19k/2.19k [00:00<00:00, 6.13MB/s]

model.safetensors: downloading bytes: █▍                    | 64.3MB, 3.81MB/s  
model.safetensors: downloading bytes: █████▏                |  229MB, 18.6MB/s  
model.safetensors: reconstructing file:  36%|██▏   |  350MB /  967MB, 6.40MB/s  
model.safetensors: download

In [4]:
!pip install -q jiwer
import re
import jiwer

reference_lines = [
    "பச்சை மாமலைபோல் மேனி",
    "பவள வாய் கமலச் செங்கண் ",
]
assert not any("PASTE" in l for l in reference_lines), "Paste your two lines first"

def norm_ta(s):
    s = re.sub(r"[^\u0B80-\u0BFF ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def srt_text_only(path):
    keep = []
    for line in open(path, encoding="utf-8").read().splitlines():
        line = line.strip()
        if not line or line.isdigit() or "-->" in line:
            continue
        keep.append(line)
    return " ".join(keep)

ref_once = norm_ta(" ".join(reference_lines))
ref_twice = norm_ta(" ".join(reference_lines * 2))

print("sanity check (should be 0.0):", jiwer.cer(ref_twice, ref_twice))

for name in ["tamil_small.srt", "tamil_medium.srt"]:
    hyp = norm_ta(srt_text_only(name))
    for label, ref in [("lines x2", ref_twice), ("lines x1", ref_once)]:
        r, h = ref.replace(" ", ""), hyp.replace(" ", "")
        print(f"{name:18} vs {label} | CER {jiwer.cer(r, h):.3f} | WER {jiwer.wer(ref, hyp):.3f}")

sanity check (should be 0.0): 0.0
tamil_small.srt    vs lines x2 | CER 0.216 | WER 0.929
tamil_small.srt    vs lines x1 | CER 1.108 | WER 1.857
tamil_medium.srt   vs lines x2 | CER 0.135 | WER 1.000
tamil_medium.srt   vs lines x1 | CER 1.081 | WER 2.143
